In [ ]:
import json
import urllib3
import boto3
from datetime import datetime

# --- CONFIGURACIÓN ---
S3_BUCKET = "proyecto-demanda-clima" 
BASE_URL = "https://apidatos.ree.es/es/datos/demanda/evolucion"

def lambda_handler(event, context):
    http = urllib3.PoolManager()
    s3 = boto3.client('s3')
    ahora = datetime.now()
    
    # Rango de fechas (desde inicio de año hasta hoy)
    start_date = f"{ahora.year}-01-01"
    end_date = ahora.strftime('%Y-%m-%d')
    
    url = f"{BASE_URL}?start_date={start_date}T00:00&end_date={end_date}T23:59&time_trunc=day"
    
    try:
        # 1. Petición a la API usando urllib3
        print(f"📡 Solicitando datos a: {url}")
        res = http.request('GET', url)
        
        if res.status == 200:
            # 2. Parsear los datos para validar que es un JSON correcto
            data = json.loads(res.data.decode('utf-8'))
            
            # 3. Definir ruta en S3 (Capa Bronce)
            file_key = f"bronce/demanda/demanda_{ahora.year}.json"
            
            # 4. Subir a S3
            s3.put_object(
                Bucket=S3_BUCKET,
                Key=file_key,
                Body=json.dumps(data, indent=4),
                ContentType='application/json'
            )
            
            mensaje = f"✅ Éxito: Datos de {ahora.year} guardados en s3://{S3_BUCKET}/{file_key}"
            print(mensaje)
            return {
                'statusCode': 200,
                'body': json.dumps(mensaje)
            }
        else:
            error_msg = f"❌ Error API REE: Código {res.status}"
            print(error_msg)
            return {
                'statusCode': res.status,
                'body': json.dumps(error_msg)
            }
            
    except Exception as e:
        print(f"💥 Error crítico: {str(e)}")
        return {
            'statusCode': 500,
            'body': json.dumps(f"Error: {str(e)}")
        }